In [ ]:
import whisper
import datetime
import os
import torch

def format_time(seconds):
    return str(datetime.timedelta(seconds=seconds)).replace(".", ",")[:11]

def create_srt_and_text(segments):
    srt_content = ""
    text_content = ""
    for i, segment in enumerate(segments, start=1):
        start_time = format_time(segment['start'])
        end_time = format_time(segment['end'])
        text = segment['text'].strip()
        srt_content += f"{i}\n{start_time} --> {end_time}\n{text}\n\n"
        text_content += f"{text}\n"
    return srt_content, text_content

def ensure_directory_exists(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

if __name__ == "__main__":
    # ディレクトリ設定
    source_dir = "./source"
    ensure_directory_exists(source_dir)
    
    # 入力ファイルのパス
    input_file = os.path.join(source_dir, "audio.mp3")
    
    print("Whisperモデルをロード中...")
    model = whisper.load_model("large")
    
    try:
        print(f"音声ファイル'{input_file}'を処理中...")
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device)
        
        # 言語自動認識で文字起こし
        result = model.transcribe(input_file, fp16=False)
        detected_language = result["language"]
        print(f"検出された言語: {detected_language}")
        
        base_name = os.path.splitext(os.path.basename(input_file))[0]
        srt_content, text_content = create_srt_and_text(result["segments"])
        
        srt_file = os.path.join(source_dir, f"{base_name}.srt")
        text_file = os.path.join(source_dir, f"{base_name}.txt")
        
        with open(srt_file, "w", encoding="utf-8") as f:
            f.write(srt_content)
        with open(text_file, "w", encoding="utf-8") as f:
            f.write(text_content)
        
        print(f"処理完了: '{srt_file}' と '{text_file}' を保存しました。")
        
    except FileNotFoundError:
        print(f"エラー: 入力ファイル '{input_file}' が見つかりません。")
    except Exception as e:
        print(f"エラーが発生しました: {str(e)}")